<div align="center">

# 🚲 BikeShare AI
### Bike-Sharing Rental Demand Prediction

**ExcelR Data Science Project | End-to-end Regression & Interactive Analytics**

*Understand demand • Explore patterns • Compare ML models • Simulate hourly rentals*

</div>

---

## Project overview

### Business objective
Predict hourly bike rental demand (`cnt`) to help a bike-sharing operator plan bike availability, reduce customer waiting time and support operational, pricing and marketing decisions.

### Project workflow
1. Business and data understanding
2. Data quality assessment and cleaning
3. Exploratory data analysis (EDA) and visualisation
4. Feature engineering and preprocessing
5. Decision Tree, Random Forest and Gradient Boosting regression
6. Hyperparameter tuning with time-series cross-validation
7. Final model evaluation and comparison
8. Interactive demand simulator and model export

### Important modelling decisions
- **Target:** `cnt`, the total number of hourly rentals.
- **Leakage prevention:** `casual` and `registered` are excluded because their sum equals `cnt`; these values would not be known when predicting future demand.
- **Index exclusion:** `instant` is a row identifier, not a useful operational predictor.
- **Time-aware validation:** the test set contains complete dates later than every training date. This avoids splitting the same day across train and test.
- **Outliers:** high demand may represent genuine rush-hour or seasonal peaks. IQR flags are investigated, not automatically deleted.
- **Source terminology:** the dataset contains the season label `springer`; it is retained as supplied rather than silently renamed.

> **How to use:** Run the notebook from top to bottom in Google Colab. Upload the supplied CSV when prompted. The model search may take a few minutes depending on the Colab runtime.

## 1. Environment setup and dataset upload

The notebook uses pandas and scikit-learn for analysis and modelling, Plotly for interactive charts, and ipywidgets for the demand simulator.

In [ ]:
# Install / enable interactive visualisation support in Colab
!pip -q install plotly ipywidgets

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

from google.colab import files, output
output.enable_custom_widget_manager()

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 10

print("Libraries loaded successfully.")

In [ ]:
# Upload the Bike-Sharing CSV file
uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith(".csv")]

if not csv_files:
    raise ValueError("No CSV file uploaded. Please upload the Bike-Sharing dataset.")

DATA_PATH = csv_files[0]
df_raw = pd.read_csv(DATA_PATH, dtype=str)

print(f"Dataset loaded: {DATA_PATH}")
print(f"Rows: {df_raw.shape[0]:,} | Columns: {df_raw.shape[1]}")
display(df_raw.head())

## 2. Business understanding and data dictionary

The dataset contains hourly bike-sharing observations. The response variable is `cnt`, which represents total rentals (casual plus registered users).

| Field | Meaning | Modelling treatment |
|---|---|---|
| `instant` | Row/index number | Excluded |
| `dteday` | Observation date | Used for chronology and date features |
| `season` | Season label | Categorical predictor |
| `yr`, `mnth`, `hr` | Year, month and hour | Calendar predictors |
| `holiday`, `weekday`, `workingday` | Calendar indicators | Categorical predictors |
| `weathersit` | Weather situation | Categorical predictor |
| `temp`, `atemp`, `hum`, `windspeed` | Normalised weather measures | Numeric predictors |
| `casual`, `registered` | Rental count components | Excluded to prevent target leakage |
| `cnt` | Total bike rentals | Prediction target |

The supplied data dictionary describes temperature, feeling temperature, humidity and wind speed as normalised measures. The weather variables are therefore used on their supplied 0–1 scales, not converted into physical units.

## 3. Data understanding and initial quality checks

In [ ]:
print("Dataset dimensions:", df_raw.shape)
print("\nColumn names:")
print(df_raw.columns.tolist())

print("\nOriginal data types:")
display(df_raw.dtypes.astype(str).rename("data_type").to_frame())

print("\nQuestion-mark values (used as missing-value markers):")
question_marks = (df_raw == "?").sum().sort_values(ascending=False)
display(question_marks[question_marks > 0].rename("question_mark_count").to_frame())

print("\nSample records:")
display(df_raw.head(10))

## 4. Data cleaning and validation

Cleaning steps:
- Replace `?` markers with missing values.
- Parse dates using the dataset's `DD-MM-YYYY` format.
- Convert numeric columns to numeric data types.
- Check duplicates, missing values, expected ranges and the arithmetic relationship between rental components and total demand.
- Keep target rows with valid `cnt` and date values. Missing predictor values are imputed inside the model pipeline, fitted on training data only.

In [ ]:
df = df_raw.replace("?", np.nan).copy()

# Dataset dates are stored as DD-MM-YYYY.
df["dteday"] = pd.to_datetime(df["dteday"], format="%d-%m-%Y", errors="coerce")

numeric_columns = [
    "instant", "yr", "mnth", "hr", "weekday",
    "temp", "atemp", "hum", "windspeed",
    "casual", "registered", "cnt"
]
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

missing_summary = df.isna().sum().sort_values(ascending=False)
print("Missing values after conversion:")
display(missing_summary[missing_summary > 0].rename("missing_count").to_frame())

print("Exact duplicate rows:", int(df.duplicated().sum()))

# Target-component consistency check
component_rows = df[["casual", "registered", "cnt"]].dropna()
component_mismatches = component_rows[
    component_rows["casual"] + component_rows["registered"] != component_rows["cnt"]
]
print("Rows where casual + registered != cnt:", len(component_mismatches))

# Range checks based on the supplied data dictionary / normalised scale.
range_rules = {
    "hr": (0, 23), "weekday": (0, 6), "mnth": (1, 12),
    "temp": (0, 1), "atemp": (0, 1), "hum": (0, 1),
    "windspeed": (0, 1), "cnt": (0, None)
}
range_results = []
for col, (lower, upper) in range_rules.items():
    invalid = df[col].notna() & (df[col] < lower)
    if upper is not None:
        invalid |= df[col].notna() & (df[col] > upper)
    range_results.append({"column": col, "out_of_range_count": int(invalid.sum())})
range_results = pd.DataFrame(range_results)
display(range_results)

# Retain rows with a usable target and date; predictors are imputed in the pipeline.
rows_before = len(df)
df = df.dropna(subset=["cnt", "dteday"]).copy()
df = df.sort_values(["dteday", "hr"]).reset_index(drop=True)

print(f"Rows retained for modelling: {len(df):,} of {rows_before:,}")
print("Date range:", df["dteday"].min().date(), "to", df["dteday"].max().date())
display(df["cnt"].describe().to_frame("hourly_rentals"))

## 5. Exploratory data analysis (EDA)

The following charts describe hourly demand, its distribution and how it varies with time, season, weather and working-day status. Plotly charts support hover, zoom and legend interaction.

In [ ]:
# KPI cards for the dataset
total_records = len(df)
mean_demand = df["cnt"].mean()
median_demand = df["cnt"].median()
peak_hour = int(df.groupby("hr")["cnt"].mean().idxmax())
peak_hour_mean = df.groupby("hr")["cnt"].mean().max()

display(HTML(f'''
<div style="display:flex;flex-wrap:wrap;gap:12px;margin:8px 0 18px 0;">
  <div style="background:#eef4ff;padding:16px 20px;border-radius:12px;min-width:150px;">
    <div style="font-size:12px;color:#475569;">Valid hourly records</div>
    <div style="font-size:26px;font-weight:700;color:#0f172a;">{total_records:,}</div>
  </div>
  <div style="background:#ecfdf5;padding:16px 20px;border-radius:12px;min-width:150px;">
    <div style="font-size:12px;color:#475569;">Average rentals / hour</div>
    <div style="font-size:26px;font-weight:700;color:#0f172a;">{mean_demand:.1f}</div>
  </div>
  <div style="background:#fff7ed;padding:16px 20px;border-radius:12px;min-width:150px;">
    <div style="font-size:12px;color:#475569;">Median rentals / hour</div>
    <div style="font-size:26px;font-weight:700;color:#0f172a;">{median_demand:.1f}</div>
  </div>
  <div style="background:#f5f3ff;padding:16px 20px;border-radius:12px;min-width:150px;">
    <div style="font-size:12px;color:#475569;">Highest average demand hour</div>
    <div style="font-size:26px;font-weight:700;color:#0f172a;">{peak_hour:02d}:00</div>
  </div>
</div>
'''))

In [ ]:
# Target distribution and outliers
fig = px.histogram(
    df, x="cnt", nbins=45, marginal="box",
    title="Distribution of hourly bike rentals",
    labels={"cnt": "Total rentals per hour"},
    opacity=0.85
)
fig.update_layout(template="plotly_white", height=480)
fig.show()

In [ ]:
# Average demand by hour
hourly = df.groupby("hr", as_index=False)["cnt"].mean()
fig = px.line(
    hourly, x="hr", y="cnt", markers=True,
    title="Average bike rental demand by hour",
    labels={"hr": "Hour of day", "cnt": "Average rentals"}
)
fig.update_xaxes(dtick=1)
fig.update_layout(template="plotly_white", height=420)
fig.show()

# Average demand by month
monthly = df.groupby("mnth", as_index=False)["cnt"].mean()
fig = px.bar(
    monthly, x="mnth", y="cnt",
    title="Average bike rental demand by month",
    labels={"mnth": "Month", "cnt": "Average rentals"}
)
fig.update_xaxes(dtick=1)
fig.update_layout(template="plotly_white", height=420)
fig.show()

In [ ]:
# Demand by season, weather and working-day status
season_order = [x for x in ["springer", "summer", "fall", "winter"] if x in df["season"].dropna().unique()]
fig = px.box(
    df.dropna(subset=["season"]), x="season", y="cnt", color="season",
    category_orders={"season": season_order},
    title="Hourly rental distribution by season",
    labels={"season": "Season (source label)", "cnt": "Hourly rentals"}
)
fig.update_layout(template="plotly_white", showlegend=False, height=450)
fig.show()

weather_mean = df.groupby("weathersit", dropna=False, as_index=False)["cnt"].mean()
fig = px.bar(
    weather_mean, x="weathersit", y="cnt", color="weathersit",
    title="Average demand by weather situation",
    labels={"weathersit": "Weather situation", "cnt": "Average rentals"}
)
fig.update_layout(template="plotly_white", showlegend=False, height=420)
fig.show()

working_mean = df.groupby("workingday", dropna=False, as_index=False)["cnt"].mean()
fig = px.bar(
    working_mean, x="workingday", y="cnt", color="workingday",
    title="Average demand: working day vs non-working day",
    labels={"workingday": "Day type", "cnt": "Average rentals"}
)
fig.update_layout(template="plotly_white", showlegend=False, height=420)
fig.show()

In [ ]:
# Daily time-series trend
daily_demand = df.groupby("dteday", as_index=False)["cnt"].mean().sort_values("dteday")
fig = px.line(
    daily_demand, x="dteday", y="cnt",
    title="Daily average hourly rental demand (2011–2012)",
    labels={"dteday": "Date", "cnt": "Average hourly rentals"}
)
fig.update_layout(template="plotly_white", height=450)
fig.show()

In [ ]:
# Correlation heatmap for numeric columns
numeric_corr = df.select_dtypes(include=np.number).drop(columns=["instant"], errors="ignore").corr()
fig = px.imshow(
    numeric_corr, text_auto=".2f", aspect="auto",
    color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
    title="Correlation matrix — numeric variables"
)
fig.update_layout(template="plotly_white", height=650)
fig.show()

### Interactive EDA explorer

Use the filters below to explore the demand pattern for a selected season, weather situation and month. The charts update when a filter changes.

In [ ]:
season_options = ["All"] + sorted(df["season"].dropna().unique().tolist())
weather_options = ["All"] + sorted(df["weathersit"].dropna().unique().tolist())
month_options = ["All"] + list(range(1, 13))

season_widget = widgets.Dropdown(options=season_options, value="All", description="Season:")
weather_widget = widgets.Dropdown(options=weather_options, value="All", description="Weather:")
month_widget = widgets.Dropdown(options=month_options, value="All", description="Month:")

eda_output = widgets.Output()

def update_eda_explorer(season, weather, month):
    with eda_output:
        clear_output(wait=True)
        filtered = df.copy()
        if season != "All":
            filtered = filtered[filtered["season"] == season]
        if weather != "All":
            filtered = filtered[filtered["weathersit"] == weather]
        if month != "All":
            filtered = filtered[filtered["mnth"] == month]

        if filtered.empty:
            print("No records match the selected filters.")
            return

        print(f"Matching observations: {len(filtered):,}")
        print(f"Average demand: {filtered['cnt'].mean():.1f} rentals/hour")
        hourly_filtered = filtered.groupby("hr", as_index=False)["cnt"].mean()
        display(px.line(
            hourly_filtered, x="hr", y="cnt", markers=True,
            title="Average hourly demand for selected filters",
            labels={"hr": "Hour", "cnt": "Average rentals"}
        ).update_layout(template="plotly_white", height=380))
        daily_filtered = filtered.groupby("dteday", as_index=False)["cnt"].mean()
        display(px.scatter(
            daily_filtered, x="dteday", y="cnt",
            title="Daily average demand for selected filters",
            labels={"dteday": "Date", "cnt": "Average rentals"}
        ).update_layout(template="plotly_white", height=350))

interactive_eda = widgets.interactive_output(
    update_eda_explorer,
    {"season": season_widget, "weather": weather_widget, "month": month_widget}
)
display(widgets.VBox([
    widgets.HBox([season_widget, weather_widget, month_widget]),
    eda_output
]))

## 6. Outlier assessment

The IQR rule is used to flag unusually low or high target values. These records are not automatically removed: a genuine rush-hour or high-season demand peak is operationally important and should remain part of the training data unless it is shown to be an error.

In [ ]:
q1, q3 = df["cnt"].quantile([0.25, 0.75])
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
outlier_mask = (df["cnt"] < lower_fence) | (df["cnt"] > upper_fence)

print(f"Q1: {q1:.2f} | Q3: {q3:.2f} | IQR: {iqr:.2f}")
print(f"Lower fence: {lower_fence:.2f} | Upper fence: {upper_fence:.2f}")
print(f"Target observations flagged by IQR: {int(outlier_mask.sum()):,}")
print("Treatment: retain valid demand peaks; investigate records only if there is evidence of data error.")

## 7. Feature engineering and leakage prevention

Calendar features are derived from the date. The dataset's weekday coding uses Sunday = 0 through Saturday = 6, so the derived weekday feature follows the same convention.

The model excludes:
- `cnt`: target.
- `casual` and `registered`: components of the target (data leakage).
- `instant`: row identifier.
- `dteday`: retained separately for chronological splitting; date-derived fields are used as predictors.

In [ ]:
model_df = df.copy()

model_df["year_from_date"] = model_df["dteday"].dt.year
model_df["month_from_date"] = model_df["dteday"].dt.month
model_df["day_of_month"] = model_df["dteday"].dt.day
# Match source weekday coding: Sunday=0, Monday=1, ..., Saturday=6.
model_df["day_of_week_from_date"] = (model_df["dteday"].dt.dayofweek + 1) % 7
model_df["is_weekend"] = model_df["day_of_week_from_date"].isin([0, 6]).astype(float)

model_df = model_df.sort_values(["dteday", "hr"]).reset_index(drop=True)

target = "cnt"
excluded_columns = ["cnt", "casual", "registered", "instant", "dteday"]
feature_columns = [c for c in model_df.columns if c not in excluded_columns]

X = model_df[feature_columns].copy()
y = model_df[target].astype(float).copy()

categorical_features = [
    "season", "yr", "mnth", "holiday", "weekday",
    "workingday", "weathersit", "hr",
    "month_from_date", "day_of_week_from_date"
]
numeric_features = [c for c in feature_columns if c not in categorical_features]

print("Target:", target)
print("Number of predictor features:", len(feature_columns))
print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)
print("Leakage columns excluded:", ["casual", "registered"])

## 8. Chronological train-test split

A random split is not suitable for this time-dependent task because it can place observations from the same date in both training and test sets. Instead, the split is performed on **unique dates**. The final 20% of dates are held out as the test period; no test date appears in training.

In [ ]:
unique_dates = np.array(sorted(model_df["dteday"].dropna().unique()))
date_split_index = int(len(unique_dates) * 0.80)
test_start_date = pd.Timestamp(unique_dates[date_split_index])

train_mask = model_df["dteday"] < test_start_date
test_mask = model_df["dteday"] >= test_start_date

X_train, X_test = X.loc[train_mask].copy(), X.loc[test_mask].copy()
y_train, y_test = y.loc[train_mask].copy(), y.loc[test_mask].copy()

train_dates = model_df.loc[train_mask, "dteday"]
test_dates = model_df.loc[test_mask, "dteday"]

assert train_dates.max() < test_dates.min(), "Chronological split check failed."
assert set(train_dates.dt.date).isdisjoint(set(test_dates.dt.date)), "A date appears in both splits."

print("Training observations:", f"{len(X_train):,}")
print("Testing observations:", f"{len(X_test):,}")
print("Training period:", train_dates.min().date(), "to", train_dates.max().date())
print("Testing period:", test_dates.min().date(), "to", test_dates.max().date())
print("Date overlap between train and test: 0")

## 9. Preprocessing and baseline model building

Missing numeric values are imputed with the training median. Missing categorical values are imputed with the training mode, then one-hot encoded. The preprocessing is part of each pipeline, so it is fitted only on the corresponding training fold.

The three algorithms required by the project brief are:
- Decision Tree Regressor
- Random Forest Regressor
- Gradient Boosting Regressor

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_preprocessor, numeric_features),
    ("cat", categorical_preprocessor, categorical_features)
])

baseline_estimators = {
    "Decision Tree": DecisionTreeRegressor(
        random_state=42, max_depth=15, min_samples_leaf=2
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, random_state=42, n_jobs=-1, min_samples_leaf=2
    ),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

def regression_metrics(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "MSE": mean_squared_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        "R2": r2_score(actual, predicted)
    }

baseline_models = {}
baseline_rows = []

for model_name, estimator in baseline_estimators.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    baseline_models[model_name] = pipeline
    baseline_rows.append({"Model": model_name, **regression_metrics(y_test, predictions)})

baseline_results = pd.DataFrame(baseline_rows).sort_values("RMSE")
display(baseline_results.round(3))

## 10. Hyperparameter tuning

`RandomizedSearchCV` is used with `TimeSeriesSplit`, preserving chronological order within each cross-validation fold. A fixed random seed makes the search reproducible. The search budget is set to eight sampled configurations per model to balance coverage and Colab runtime.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import randint, uniform

time_cv = TimeSeriesSplit(n_splits=3)

parameter_spaces = {
    "Decision Tree": {
        "model__max_depth": [5, 8, 12, 16, 20, None],
        "model__min_samples_split": randint(2, 20),
        "model__min_samples_leaf": randint(1, 10),
        "model__max_features": [None, "sqrt", 0.8]
    },
    "Random Forest": {
        "model__n_estimators": randint(100, 300),
        "model__max_depth": [8, 12, 16, 20, None],
        "model__min_samples_split": randint(2, 20),
        "model__min_samples_leaf": randint(1, 8),
        "model__max_features": ["sqrt", 0.8, 1.0]
    },
    "Gradient Boosting": {
        "model__n_estimators": randint(80, 220),
        "model__learning_rate": uniform(0.02, 0.18),
        "model__max_depth": randint(2, 5),
        "model__min_samples_leaf": randint(1, 8),
        "model__subsample": uniform(0.7, 0.3)
    }
}

tuned_searches = {}
tuning_rows = []

for model_name, estimator in baseline_estimators.items():
    print(f"Searching hyperparameters: {model_name}")
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=parameter_spaces[model_name],
        n_iter=8,
        scoring="neg_root_mean_squared_error",
        cv=time_cv,
        random_state=42,
        n_jobs=-1,
        verbose=0,
        refit=True
    )
    search.fit(X_train, y_train)
    tuned_searches[model_name] = search
    tuning_rows.append({
        "Model": model_name,
        "Best CV RMSE": -search.best_score_,
        "Best parameters": search.best_params_
    })

tuning_results = pd.DataFrame(tuning_rows).sort_values("Best CV RMSE")
display(tuning_results[["Model", "Best CV RMSE"]].round(3))

### Best hyperparameters found

The table below makes the selected configuration for each algorithm visible and auditable.

In [ ]:
for model_name, search in tuned_searches.items():
    print("\n" + "=" * 75)
    print(model_name)
    print("Best time-series CV RMSE:", round(-search.best_score_, 3))
    print("Best parameters:")
    for parameter, value in search.best_params_.items():
        print(f"  {parameter}: {value}")

## 11. Final model evaluation and comparison

The tuned models are evaluated on the same held-out chronological test period. The primary selection metric is RMSE; MAE and R² are also reported to provide a fuller view of model performance.

In [ ]:
final_rows = []
final_predictions = {}

for model_name, search in tuned_searches.items():
    model = search.best_estimator_
    predictions = model.predict(X_test)
    final_predictions[model_name] = predictions
    final_rows.append({"Model": model_name, **regression_metrics(y_test, predictions)})

final_results = pd.DataFrame(final_rows).sort_values("RMSE").reset_index(drop=True)
display(final_results.round(3))

best_model_name = final_results.loc[0, "Model"]
best_model = tuned_searches[best_model_name].best_estimator_
best_predictions = final_predictions[best_model_name]

best_metrics = regression_metrics(y_test, best_predictions)
print(f"Selected model: {best_model_name}")
print(f"Test MAE:  {best_metrics['MAE']:.3f} rentals/hour")
print(f"Test RMSE: {best_metrics['RMSE']:.3f} rentals/hour")
print(f"Test R²:   {best_metrics['R2']:.4f}")

In [ ]:
# Interactive comparison of test-set metrics
metrics_long = final_results.melt(
    id_vars="Model",
    value_vars=["MAE", "RMSE"],
    var_name="Metric",
    value_name="Error"
)
fig = px.bar(
    metrics_long, x="Model", y="Error", color="Metric", barmode="group",
    title="Tuned model comparison — MAE and RMSE (lower is better)",
    text_auto=".1f"
)
fig.update_layout(template="plotly_white", height=450)
fig.show()

fig = px.bar(
    final_results, x="Model", y="R2", color="Model",
    title="Tuned model comparison — R² (higher is better)",
    text_auto=".3f"
)
fig.update_layout(template="plotly_white", showlegend=False, height=420)
fig.show()

## 12. Prediction diagnostics

The actual-versus-predicted chart shows how close predictions are to the ideal diagonal. The residual chart shows the distribution of prediction errors. The chronological line chart compares actual and predicted demand for a section of the test period.

In [ ]:
diagnostics = pd.DataFrame({
    "Date": test_dates.values,
    "Actual": y_test.values,
    "Predicted": best_predictions,
    "Residual": y_test.values - best_predictions
})

fig = px.scatter(
    diagnostics, x="Actual", y="Predicted",
    title=f"Actual vs predicted hourly demand — {best_model_name}",
    opacity=0.45,
    labels={"Actual": "Actual rentals", "Predicted": "Predicted rentals"}
)
axis_min = min(diagnostics["Actual"].min(), diagnostics["Predicted"].min())
axis_max = max(diagnostics["Actual"].max(), diagnostics["Predicted"].max())
fig.add_trace(go.Scatter(
    x=[axis_min, axis_max], y=[axis_min, axis_max],
    mode="lines", name="Ideal prediction", line=dict(dash="dash")
))
fig.update_layout(template="plotly_white", height=500)
fig.show()

fig = px.histogram(
    diagnostics, x="Residual", nbins=45, marginal="box",
    title="Residual distribution",
    labels={"Residual": "Actual minus predicted rentals"}
)
fig.update_layout(template="plotly_white", height=450)
fig.show()

diagnostics_sample = diagnostics.head(300)
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=diagnostics_sample["Date"], y=diagnostics_sample["Actual"],
    mode="lines", name="Actual"
))
fig.add_trace(go.Scatter(
    x=diagnostics_sample["Date"], y=diagnostics_sample["Predicted"],
    mode="lines", name="Predicted"
))
fig.update_layout(
    title="Actual vs predicted demand — first 300 test observations",
    xaxis_title="Date", yaxis_title="Hourly rentals",
    template="plotly_white", height=480
)
fig.show()

## 13. Feature importance

For the selected tree-based model, feature importance is extracted after preprocessing. One-hot encoded category levels appear as separate features. Importance describes how the fitted model used the features; it does not establish causation.

In [ ]:
fitted_preprocessor = best_model.named_steps["preprocessor"]
fitted_estimator = best_model.named_steps["model"]
encoded_feature_names = fitted_preprocessor.get_feature_names_out()
feature_importances = getattr(fitted_estimator, "feature_importances_", None)

if feature_importances is not None:
    importance_table = pd.DataFrame({
        "Feature": encoded_feature_names,
        "Importance": feature_importances
    }).sort_values("Importance", ascending=False).head(20)

    display(importance_table.round(4))
    fig = px.bar(
        importance_table.sort_values("Importance"),
        x="Importance", y="Feature", orientation="h",
        title=f"Top 20 feature importances — {best_model_name}"
    )
    fig.update_layout(template="plotly_white", height=650)
    fig.show()
else:
    print("The selected estimator does not expose feature_importances_.")

## 14. Interactive bike-demand prediction simulator

Choose a scenario and click **Predict hourly demand**. The controls build a new observation using the same feature columns and preprocessing pipeline used during training.

**Input note:** temperature, feeling temperature, humidity and wind speed use the dataset's normalised 0–1 scales. The date selector is for scenario construction; the training dataset covers 2011–2012, so predictions for dates far outside that historical period should be treated as extrapolations, not validated forecasts.

In [ ]:
# Map month to the season labels observed in this supplied dataset.
# This follows the dominant month-season relationship in the source data.
month_to_season = {
    1: "springer", 2: "springer", 3: "springer",
    4: "summer", 5: "summer", 6: "summer",
    7: "fall", 8: "fall", 9: "fall",
    10: "winter", 11: "winter", 12: "winter"
}
season_display = {
    "springer": "Spring (source label: springer)",
    "summer": "Summer",
    "fall": "Fall",
    "winter": "Winter"
}
season_reverse = {v: k for k, v in season_display.items()}

prediction_date = widgets.DatePicker(
    description="Scenario date:",
    value=pd.Timestamp("2012-08-07").date(),
    layout=widgets.Layout(width="300px")
)
prediction_hour = widgets.IntSlider(
    value=8, min=0, max=23, step=1, description="Hour:",
    continuous_update=False, style={"description_width": "initial"}
)
prediction_weather = widgets.Dropdown(
    options=["Clear", "Mist", "Light Snow", "Heavy Rain"],
    value="Clear", description="Weather:"
)
prediction_season = widgets.Dropdown(
    options=[season_display[s] for s in ["springer", "summer", "fall", "winter"]],
    value=season_display["fall"], description="Season:"
)
prediction_holiday = widgets.Dropdown(
    options=["No", "Yes"], value="No", description="Holiday:"
)
prediction_workingday = widgets.Dropdown(
    options=["Working Day", "No work"], value="Working Day",
    description="Day type:"
)
prediction_temp = widgets.FloatSlider(
    value=0.60, min=0.0, max=1.0, step=0.01, description="Temperature:",
    readout_format=".2f", continuous_update=False
)
prediction_atemp = widgets.FloatSlider(
    value=0.60, min=0.0, max=1.0, step=0.01, description="Feels-like temp:",
    readout_format=".2f", continuous_update=False
)
prediction_humidity = widgets.FloatSlider(
    value=0.55, min=0.0, max=1.0, step=0.01, description="Humidity:",
    readout_format=".2f", continuous_update=False
)
prediction_wind = widgets.FloatSlider(
    value=0.20, min=0.0, max=1.0, step=0.01, description="Wind speed:",
    readout_format=".2f", continuous_update=False
)

predict_button = widgets.Button(
    description="Predict hourly demand",
    button_style="success",
    icon="bicycle",
    layout=widgets.Layout(width="240px", height="42px")
)
prediction_output = widgets.Output()

def build_scenario_row():
    if prediction_date.value is None:
        raise ValueError("Please choose a scenario date.")

    date_value = pd.Timestamp(prediction_date.value)
    weekday_code = (date_value.dayofweek + 1) % 7
    season_value = season_reverse[prediction_season.value]

    row = {
        "season": season_value,
        "yr": date_value.year,
        "mnth": date_value.month,
        "hr": prediction_hour.value,
        "holiday": prediction_holiday.value,
        "weekday": weekday_code,
        "workingday": prediction_workingday.value,
        "weathersit": prediction_weather.value,
        "temp": prediction_temp.value,
        "atemp": prediction_atemp.value,
        "hum": prediction_humidity.value,
        "windspeed": prediction_wind.value,
        "year_from_date": date_value.year,
        "month_from_date": date_value.month,
        "day_of_month": date_value.day,
        "day_of_week_from_date": weekday_code,
        "is_weekend": float(weekday_code in [0, 6])
    }

    scenario = pd.DataFrame([row], columns=feature_columns)
    return scenario, date_value

def run_prediction(_):
    with prediction_output:
        clear_output(wait=True)
        try:
            scenario, scenario_date = build_scenario_row()
            prediction = float(best_model.predict(scenario)[0])
            prediction = max(0, prediction)
            display(HTML(f'''
            <div style="background:#ecfdf5;border:1px solid #a7f3d0;border-radius:14px;padding:20px;margin-top:10px;">
              <div style="font-size:13px;color:#065f46;">Predicted hourly bike rentals</div>
              <div style="font-size:38px;font-weight:750;color:#064e3b;">{prediction:,.0f}</div>
              <div style="font-size:13px;color:#065f46;">{scenario_date.strftime("%d %b %Y")} at {prediction_hour.value:02d}:00 · {prediction_weather.value}</div>
            </div>
            '''))
            if scenario_date.year not in [2011, 2012]:
                print("Caution: this date is outside the model's 2011–2012 training period.")
            print("Selected model:", best_model_name)
            print("This is a model estimate, not a guaranteed number of available bikes.")
        except Exception as error:
            print("Prediction could not be generated:", error)

predict_button.on_click(run_prediction)

prediction_controls = widgets.VBox([
    widgets.HBox([prediction_date, prediction_hour]),
    widgets.HBox([prediction_season, prediction_weather]),
    widgets.HBox([prediction_holiday, prediction_workingday]),
    prediction_temp,
    prediction_atemp,
    prediction_humidity,
    prediction_wind,
    predict_button,
    prediction_output
])
display(prediction_controls)

## 15. Model export and deployment-ready inference

The complete fitted pipeline is saved, including imputers, one-hot encoder and estimator. The exported bundle also stores the expected feature-column order and model name.

This provides a reusable inference artifact. The notebook simulator above demonstrates inference in Colab; production hosting, API integration, monitoring and scheduled retraining would be separate deployment activities.

In [ ]:
import joblib
from pathlib import Path

MODEL_FILE = "bike_rental_demand_model.joblib"
model_bundle = {
    "model": best_model,
    "feature_columns": feature_columns,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "model_name": best_model_name,
    "training_date_start": str(train_dates.min().date()),
    "training_date_end": str(train_dates.max().date()),
    "test_date_start": str(test_dates.min().date()),
    "test_date_end": str(test_dates.max().date()),
    "test_metrics": best_metrics
}
joblib.dump(model_bundle, MODEL_FILE)
print("Model artifact saved:", MODEL_FILE)

# Demonstrate loading the exported artifact and reproducing a prediction.
loaded_bundle = joblib.load(MODEL_FILE)
loaded_model = loaded_bundle["model"]
sample_prediction = loaded_model.predict(X_test.iloc[[0]])[0]
print("Loaded model:", loaded_bundle["model_name"])
print("Sample prediction from reloaded artifact:", round(float(sample_prediction), 2))

files.download(MODEL_FILE)

## 16. Final project findings and business interpretation

The following summary is generated from the notebook's actual run, so it stays aligned with the data and model results.

In [ ]:
# Data quality summary
missing_total = int(df_raw.replace("?", np.nan).isna().sum().sum())
duplicate_total = int(df_raw.duplicated().sum())
component_mismatch_total = int(len(component_mismatches))

# EDA insights
hourly_means = df.groupby("hr")["cnt"].mean()
peak_hour_value = int(hourly_means.idxmax())
lowest_hour_value = int(hourly_means.idxmin())
season_means = df.groupby("season")["cnt"].mean().sort_values(ascending=False)
weather_means = df.groupby("weathersit")["cnt"].mean().sort_values(ascending=False)
month_means = df.groupby("mnth")["cnt"].mean().sort_values(ascending=False)

print("PROJECT CONCLUSION")
print("=" * 72)
print(f"Dataset: {len(df):,} valid observations across {df['dteday'].nunique():,} dates.")
print(f"Question-mark / missing entries before imputation: {missing_total:,}.")
print(f"Exact duplicate rows: {duplicate_total}.")
print(f"Rows with a mismatch in casual + registered = cnt: {component_mismatch_total}.")
print(f"Average hourly demand: {df['cnt'].mean():.2f}; median: {df['cnt'].median():.2f}.")
print(f"Highest average demand hour: {peak_hour_value:02d}:00.")
print(f"Lowest average demand hour: {lowest_hour_value:02d}:00.")
print(f"Highest average-demand season in this dataset: {season_means.index[0]} ({season_means.iloc[0]:.2f}).")
print(f"Highest average-demand weather category: {weather_means.index[0]} ({weather_means.iloc[0]:.2f}).")
print(f"Highest average-demand month: {int(month_means.index[0])} ({month_means.iloc[0]:.2f}).")
print()
print("MODEL RESULTS ON THE CHRONOLOGICAL HOLDOUT")
display(final_results.round(3))
print(f"Selected model: {best_model_name}")
print(f"MAE: {best_metrics['MAE']:.2f} rentals/hour")
print(f"RMSE: {best_metrics['RMSE']:.2f} rentals/hour")
print(f"R²: {best_metrics['R2']:.4f}")

### Business recommendations

1. **Bike allocation:** use hourly demand estimates to plan redistribution before expected high-demand periods.
2. **Staffing and operations:** align field teams and maintenance coverage with recurring demand peaks.
3. **Weather-aware planning:** monitor demand changes across weather categories and prepare operational contingencies.
4. **Pricing and promotions:** use forecasts as one input to evaluate targeted pricing or marketing experiments; do not treat model associations as causal effects.
5. **Monitoring:** compare forecasts with actual rentals over time. Investigate sustained error increases and retrain when demand patterns or operating conditions change.

### Limitations

- The dataset covers 2011–2012 and does not include station-level inventory, station location, bike availability, trip duration or live operational data.
- The target is hourly system-wide rental count, not demand at an individual docking station.
- The model's test score is measured on a historical holdout. Performance on current demand patterns must be validated before operational use.
- Predictions for dates outside the 2011–2012 training period are extrapolations.
- Feature importance and observed relationships do not prove causation.
- The notebook exports a deployment-ready model pipeline and demonstrates interactive inference in Colab; it is not itself a continuously hosted production service.

### Final conclusion

The project builds a complete regression workflow to estimate hourly bike rental demand using calendar, season and weather information. Decision Tree, Random Forest and Gradient Boosting models are compared, tuned with time-series cross-validation and evaluated on a later chronological test period. The best-performing tuned model is selected using test RMSE and exported as a reusable pipeline. The interactive EDA explorer and demand simulator demonstrate how analysts can inspect patterns and test demand scenarios. The forecasts can support operational planning, subject to ongoing monitoring and validation on newer data.

## 17. Submission checklist

Before submitting:
- [ ] Run all cells from the beginning in a fresh Colab runtime.
- [ ] Confirm the CSV upload and data-quality checks complete without errors.
- [ ] Check that the train and test date ranges do not overlap.
- [ ] Confirm all three tuned models appear in the final comparison table.
- [ ] Test the interactive EDA filters.
- [ ] Test the demand simulator with at least two different scenarios.
- [ ] Confirm the `.joblib` file is generated and downloaded.
- [ ] Save a final copy of the executed notebook (`File → Download → Download .ipynb`).
- [ ] Be prepared to explain the leakage prevention, chronological split, metrics, model choice and deployment limitations.

**ExcelR project alignment:** This notebook addresses the supplied brief's EDA, cleaning, visualisation, feature engineering, Decision Tree / Random Forest / Gradient Boosting regression, hyperparameter tuning, evaluation and model-export / inference demonstration requirements. The supplied participation guidelines also expect active contribution, regular communication and weekly progress presentations.